# Notebook 02: Web Scraping & Text Extraction

**Time:** 30 minutes  
**Prerequisites:** Notebook 01 complete  
**Goal:** Extract clean text from web pages using traditional and modern tools

This notebook will:
1. Extract text from web pages with trafilatura (traditional approach)
2. Use Crawl4AI for LLM-ready markdown extraction (modern 2025 approach)
3. Scrape arXiv paper abstracts for a pretraining dataset
4. Compare extraction quality between tools

> **Why this matters:** The quality of LLM pretraining data starts with extraction. Garbage in, garbage out. Modern tools like Crawl4AI (50K+ GitHub stars) produce cleaner, LLM-optimized markdown directly, while trafilatura remains a solid baseline for simpler extraction tasks.

In [1]:
import os, sys, time, importlib
from pathlib import Path

notebook_dir = os.getcwd()
parent_dir   = str(Path(notebook_dir).parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from dotenv import load_dotenv
load_dotenv(os.path.join(parent_dir, '.env'), override=True)

import src.llm_client, src.cost_tracker, src.utils, src.config
for mod in [src.llm_client, src.cost_tracker, src.utils, src.config]:
    importlib.reload(mod)

from src.llm_client import LLMClient
from src.cost_tracker import CostTracker
from src.utils import format_response, append_to_reflection
import src.config as config

# Week 3 specific imports
import src.scraping_utils
importlib.reload(src.scraping_utils)
from src.scraping_utils import (
    extract_with_trafilatura,
    scrape_arxiv_abstracts,
    compare_extractors,
)

client  = LLMClient(path=config.PATH)
tracker = CostTracker()

outputs_dir = os.path.join('..', 'outputs')
os.makedirs(outputs_dir, exist_ok=True)

print("Setup complete -- ready for Notebook 02")

✓ Claude API client initialized
  Default model: claude-sonnet-4-6
  Available: claude-sonnet-4-6, claude-opus-4-6, claude-haiku-4-5-20251001
Setup complete -- ready for Notebook 02


---

## Part 1: Common Crawl & trafilatura

Common Crawl is a massive public archive of web pages -- like a snapshot of the entire internet, updated monthly. It's used to train models like GPT, Claude, and LLaMA.

**trafilatura** is a Python library that extracts the main content from web pages, stripping away navigation, ads, and boilerplate. Think of it as using a highlighter on millions of web pages to extract just the important sentences.

In [3]:
print("=" * 65)
print("Experiment 1: trafilatura Text Extraction")
print("=" * 65)
print()

# Extract text from an arXiv abstract page
url = "https://arxiv.org/abs/2404.00001"
result = extract_with_trafilatura(url)

print(f"\nExtracted text preview:")
print(result['text'][:500] if result['text'] else 'No text extracted')

Experiment 1: trafilatura Text Extraction

[trafilatura] Fetching: https://arxiv.org/abs/2404.00001
  Extracted 2,434 chars in 0.2s

Extracted text preview:
Physics > Physics Education
[Submitted on 3 Feb 2024]
Title:Uso de herramientas digitales matemáticas en la Educación Secundaria
View PDF HTML (experimental)Abstract:Information and Community Technologies (ICT) are very present in our society nowadays and particularly in the educative field. In just two decades, we have passed from a learning based, in many cases, on the master lessons to one such that methodologies like the flipped classroom or the gamification are stronger than ever. Along thi


In [4]:
print("=" * 65)
print("Experiment 2: Extract from a Blog Post")
print("=" * 65)
print()

# Try a blog/news article
blog_url = "https://ai.meta.com/blog/meta-llama-4/"
blog_result = extract_with_trafilatura(blog_url)

print(f"\nExtracted {blog_result['char_count']:,} chars")
print(f"Preview: {blog_result['text'][:400]}..." if blog_result['text'] else 'No text extracted')

Experiment 2: Extract from a Blog Post

[trafilatura] Fetching: https://ai.meta.com/blog/meta-llama-4/
  Extracted 120 chars in 0.7s

Extracted 120 chars
Preview: You might find what you need on our homepage .
Foundational models
Our approach
Research
Meta AI
Latest news
Meta © 2026...


In [5]:
# TODO 1: Extract text from a URL of your choice
#
# Pick a web page related to your interests (blog post, docs page, wiki article).
# Extract the text and evaluate the quality.

# my_url = "[STUDENT: PASTE YOUR URL HERE]"
my_url = "https://github.com/facebookresearch/tribev2"

print("=" * 65)
print("TODO 1: Custom URL Extraction")
print("=" * 65)
print()

my_result = extract_with_trafilatura(my_url)
print(f"\nExtracted {my_result['char_count']:,} chars")
print(f"Preview: {my_result['text'][:300]}..." if my_result['text'] else 'No text extracted')

todo1_reflection = """
[YOUR REFLECTION HERE]

- What URL did you choose and why?
- How well did trafilatura extract the main content?
- Were there any missing or incorrect parts in the extraction?
"""

print()
print(todo1_reflection)

TODO 1: Custom URL Extraction

[trafilatura] Fetching: https://github.com/facebookresearch/tribev2
  Extracted 2,959 chars in 1.0s

Extracted 2,959 chars
Preview: A Foundation Model of Vision, Audition, and Language for In-Silico Neuroscience
TRIBE v2 is a deep multimodal brain encoding model that predicts fMRI brain responses to naturalistic stimuli (video, audio, text). It combines state-of-the-art text, audio and video models into a unified Transformer arc...


[YOUR REFLECTION HERE]

- What URL did you choose and why?
- How well did trafilatura extract the main content?
- Were there any missing or incorrect parts in the extraction?



---

## Part 2: Markdown Extraction -- Crawl4AI & html2text (2025)

The key insight of modern web scraping for LLMs: **output Markdown, not plain text**. Structured Markdown preserves headings, links, lists, and code blocks -- all of which help LLMs understand document structure.

**Crawl4AI** (50K+ GitHub stars) is the leading tool for this. It uses a headless browser to render JavaScript-heavy pages and outputs clean, LLM-ready Markdown. However, it requires **Python 3.10+**.

For Python 3.9 environments, we use **html2text** as a fallback -- it converts HTML to Markdown without a browser, which works great for static pages like arXiv.

| Tool | Output | JS Support | Python | Best For |
|------|--------|-----------|--------|----------|
| **trafilatura** | Plain text | No | 3.7+ | Bulk extraction, removing boilerplate |
| **html2text** | Markdown | No | 3.6+ | Static pages, lightweight Markdown |
| **Crawl4AI** | Markdown | Yes (browser) | 3.10+ | JS-heavy sites, LLM pipelines |

```
pip install html2text        # Lightweight fallback (works everywhere)
pip install crawl4ai         # Full-featured (requires Python 3.10+)
```

In [7]:
print("=" * 65)
print("Experiment 3: Markdown Extraction vs trafilatura")
print("=" * 65)
print()

# Compare both extractors on the same page
# Uses Crawl4AI if Python 3.10+, otherwise html2text as markdown fallback
comparison_url = "https://arxiv.org/abs/2404.00001"

try:
    comparison = compare_extractors(comparison_url)
    
    print("\n--- trafilatura output (plain text, first 300 chars) ---")
    print(comparison['trafilatura']['text'][:300])
    
    md_result = comparison['crawl4ai']
    print(f"\n--- {md_result.get('method', 'markdown')} output (first 300 chars) ---")
    print(md_result['text'][:300])
    
    print(f"\nKey difference: trafilatura gives plain text ({comparison['trafilatura']['char_count']:,} chars)")
    print(f"Markdown extractor preserves structure ({md_result['char_count']:,} chars)")
except Exception as e:
    print(f"Comparison failed: {e}")
    print("TIP: pip install html2text  (or pip install crawl4ai for Python 3.10+)")

Experiment 3: Markdown Extraction vs trafilatura

EXTRACTOR COMPARISON
URL: https://arxiv.org/abs/2404.00001

[trafilatura] Fetching: https://arxiv.org/abs/2404.00001
  Extracted 2,434 chars in 0.2s
[Crawl4AI] Fetching: https://arxiv.org/abs/2404.00001


[INIT].... → Crawl4AI 0.8.6 

[FETCH]... ↓ https://arxiv.org/abs/2404.00001                                                                     |
✓ | ⏱: 0.83s 

[SCRAPE].. ◆ https://arxiv.org/abs/2404.00001                                                                     |
✓ | ⏱: 0.01s 

[COMPLETE] ● https://arxiv.org/abs/2404.00001                                                                     |
✓ | ⏱: 0.85s 

  Extracted 8,714 chars in 3.2s

--- Comparison ---
  trafilatura:  2,434 chars in 0.2s
  crawl4ai: 8,714 chars in 3.2s

--- trafilatura output (plain text, first 300 chars) ---
Physics > Physics Education
[Submitted on 3 Feb 2024]
Title:Uso de herramientas digitales matemáticas en la Educación Secundaria
View PDF HTML (experimental)Abstract:Information and Community Technologies (ICT) are very present in our society nowadays and particularly in the educative field. In just

--- crawl4ai output (first 300 chars) ---
[Skip to main content](https://arxiv.org/abs/2404.00001#content)
[![Cornell University](https://arxiv.org/static/browse/0.3.4/images/icons/cu/cornell-reduced-white-SMALL.svg)](https://www.cornell.edu/)
[Learn about arXiv becoming an independent nonprofit.](https://tech.cornell.edu/arxiv/)
We gratefu

Key difference: trafilatura gives plain text (2,434 chars)
Markdown extractor preserves structure (8,714 chars)


In [8]:
# TODO 2: Analyze the differences between extractors
#
# Use the LLM to analyze the quality differences between
# trafilatura and Crawl4AI outputs.

print("=" * 65)
print("TODO 2: LLM Analysis of Extraction Quality")
print("=" * 65)
print()

traf_text = blog_result['text'][:500] if blog_result['text'] else 'No extraction'

start = time.time()
response = client.generate(
    prompt=f"""Compare these two web extraction approaches for building LLM pretraining datasets:

1. **trafilatura** (traditional, 2020): Extracts plain text from HTML, removes boilerplate.
   Sample output: {traf_text[:200]}

2. **Crawl4AI** (modern, 2025): Produces LLM-ready Markdown with structure preserved.

For a team building a pretraining dataset:
- Which tool would you recommend for different scenarios?
- What are the trade-offs (speed, quality, features)?
- When would you use one over the other?""",
    system="You are an expert in data engineering for LLM pretraining.",
    max_tokens=400,
    temperature=0.5
)
elapsed = time.time() - start

if "error" not in response:
    tracker.add_call(response)
    print(f"Response in {elapsed:.1f}s")
    print(format_response(response, verbose=True))
else:
    print(f"Error: {response['error']}")

todo2_reflection = """
[YOUR REFLECTION HERE]

- Which extractor produced better output for your test URLs?
- For what types of pages would you prefer Crawl4AI over trafilatura?
- How does output format (plain text vs Markdown) affect downstream LLM usage?
"""

print()
print(todo2_reflection)

TODO 2: LLM Analysis of Extraction Quality

Response in 10.3s
Model: claude-sonnet-4-6
Tokens: 182 in, 400 out
Stop reason: max_tokens
# Web Extraction for LLM Pretraining: trafilatura vs Crawl4AI

## The Core Problem Your Sample Reveals

Your trafilatura output demonstrates the fundamental issue clearly:

```
You might find what you need on our homepage .
Foundational models
Our approach
Research
Meta AI
Latest news
Meta © 2026
```

**This is navigation/boilerplate leakage** — exactly what you *don't* want in pretraining data. The extractor failed to distinguish substantive content from site chrome. This isn't a trafilatura limitation per se — it's a signal that the page had minimal article-style content, and the tool made a reasonable but wrong call.

---

## Honest Tool Comparison

### trafilatura (2020)

**What it actually does well:**
- Designed specifically for **article/blog extraction** — news sites, Wikipedia-style pages
- Fast: ~1000+ pages/minute on modest hardware
- Battle-

---

## Part 3: Building an arXiv Paper Dataset

Let's build a small pretraining dataset by scraping scientific paper abstracts from arXiv. This mirrors what real pretraining pipelines do at scale -- Meta's LLaMA 4 used academic papers as a key data source.

In [9]:
print("=" * 65)
print("Experiment 4: Scrape arXiv Papers")
print("=" * 65)
print()

papers = scrape_arxiv_abstracts(
    topic="large language models",
    max_results=5,
    save_path=os.path.join(outputs_dir, 'arxiv_papers.json'),
)

Experiment 4: Scrape arXiv Papers

Scraping arXiv: 'large language models' (max 5 papers)

  [1] Vanast: Virtual Try-On with Human Image Animation via Synthetic Triplet Supervis...
      Authors: Hyunsoo Cha, Wonjung Woo, Byungjun Kim...
      Abstract: We present Vanast, a unified framework that generates garment-transferred human animation videos directly from a single ...

  [2] Beyond the Final Actor: Modeling the Dual Roles of Creator and Editor for Fine-G...
      Authors: Yang Li, Qiang Sheng, Zhengjia Wang...
      Abstract: The misuse of large language models (LLMs) requires precise detection of synthetic text. Existing works mainly follow bi...

  [3] LoMa: Local Feature Matching Revisited...
      Authors: David Nordström, Johan Edstedt, Georg Bökman...
      Abstract: Local feature matching has long been a fundamental component of 3D vision systems such as Structure-from-Motion (SfM), y...

  [4] Early Stopping for Large Reasoning Models via Confidence Dynamics...
      Aut

In [10]:
# TODO 3: Scrape papers on YOUR topic
#
# Choose a topic relevant to your capstone project or interests.
# Scrape 5-10 papers and save the results.

my_topic = "[STUDENT: ENTER YOUR TOPIC HERE]"  # e.g., "AI safety", "robotics", "NLP"

print("=" * 65)
print("TODO 3: Custom arXiv Scraping")
print("=" * 65)
print()

my_papers = scrape_arxiv_abstracts(
    topic=my_topic,
    max_results=8,
    save_path=os.path.join(outputs_dir, 'my_arxiv_papers.json'),
)

# Ask Claude to analyze the papers
paper_titles = [p['title'] for p in my_papers]
start = time.time()
response = client.generate(
    prompt=f"""I scraped these {len(my_papers)} papers from arXiv on the topic '{my_topic}':

{chr(10).join(f'{i+1}. {t}' for i, t in enumerate(paper_titles))}

Briefly analyze: What themes do you see? Which 2-3 papers seem most relevant for understanding current trends in this area?""",
    system="You are a research assistant helping analyze academic literature.",
    max_tokens=400,
    temperature=0.5
)
elapsed = time.time() - start

if "error" not in response:
    tracker.add_call(response)
    print(f"\nAnalysis ({elapsed:.1f}s):")
    print(format_response(response, verbose=False))

todo3_reflection = """
[YOUR REFLECTION HERE]

- What topic did you choose and why?
- Were you surprised by any of the papers found?
- How could this scraping approach scale to build a real pretraining dataset?
"""

print()
print(todo3_reflection)

TODO 3: Custom arXiv Scraping

Scraping arXiv: '[STUDENT: ENTER YOUR TOPIC HERE]' (max 8 papers)

  [1] Your Pre-trained Diffusion Model Secretly Knows Restoration...
      Authors: Sudarshan Rajagopalan, Vishal M. Patel
      Abstract: Pre-trained diffusion models have enabled significant advancements in All-in-One Restoration (AiOR), offering improved p...

  [2] How AI Aggregation Affects Knowledge...
      Authors: Daron Acemoglu, Tianyi Lin, Asuman Ozdaglar...
      Abstract: Artificial intelligence (AI) changes social learning when aggregated outputs become training data for future predictions...

  [3] Connection between the contextuality breaking and incompatibility breaking qubit...
      Authors: Swati Kumari, Sumit Mukherjee, R. Prabhu
      Abstract: Contextuality and measurement incompatibility are two fundamental aspects of nonclassicality, and their manifestations i...

  [4] Topological surface states revealed by the Zeeman effect in superconducting UTe2...
      Author

---

## Summary & Reflection

In [11]:
_todo1 = todo1_reflection.strip() if 'todo1_reflection' in dir() else '[TODO 1 not completed yet]'
_todo2 = todo2_reflection.strip() if 'todo2_reflection' in dir() else '[TODO 2 not completed yet]'
_todo3 = todo3_reflection.strip() if 'todo3_reflection' in dir() else '[TODO 3 not completed yet]'

full_reflection = f"""
### Part 1 - trafilatura Extraction

{_todo1}

---

### Part 2 - Crawl4AI vs trafilatura

{_todo2}

---

### Part 3 - arXiv Paper Dataset

{_todo3}
"""

reflection_file = append_to_reflection(
    notebook="02",
    section_title="Web Scraping & Text Extraction",
    reflection_content=full_reflection,
    output_dir=os.path.join('..', 'outputs')
)

print(f"Reflection saved: {reflection_file}")
print()
tracker.report()

Reflection saved: ../outputs/homework_reflection.md

API COST REPORT
Total API calls:     2
Total input tokens:  427
Total output tokens: 723
Total cost:          $0.0121

Last 2 calls:
  1. [00:52:28] sonnet -- 182in/400out -- $0.0065
  2. [00:55:10] sonnet -- 245in/323out -- $0.0056


## Notebook 02 Complete!

**What you accomplished:**
- Extracted clean text from web pages with trafilatura
- Compared traditional vs modern (Crawl4AI) extraction approaches
- Built a mini paper dataset from arXiv

**Key concepts:**
- trafilatura removes HTML boilerplate to extract main content
- Crawl4AI produces LLM-ready Markdown with preserved structure
- arXiv API provides structured access to scientific papers

**Next:** Open **Notebook 03: Document OCR & PDF Extraction**